# Demo — Extending the Define-XML Model

Vendor extensions in a custom namespace are a designed-in part of ODM. In odmlib, extension
is plain Python subclassing — the same mechanism odmlib itself uses (`define_2_1` extends
`odm_1_3_2`; `arm_1_0` extends `define_2_1`).

This demo adds a `vnd:ReviewStatus` attribute to `ItemDef`, writes an extended define.xml,
and round-trips it. See `lecture_notes.md` for the full discussion.

In [ ]:
import os
import odmlib.define_loader as DL
import odmlib.loader as LD
import odmlib.define_2_1.model as DEF
import odmlib.ns_registry as NS
import odmlib.typed as T

os.makedirs("output", exist_ok=True)

## 1. Extend: three lines

Register the vendor namespace, then subclass with `merge_fields=True` to inherit every base
field without redeclaring them.

**Name the subclass after the element** — the XML tag comes from the class name, so shadowing
the stock `ItemDef` name is exactly what we want. (And never register an extension namespace
as the default — that belongs to ODM.)

In [ ]:
NS.NamespaceRegistry(prefix="vnd", uri="https://example.org/vnd/v1.0")

class ItemDef(DEF.ItemDef, merge_fields=True):
    ReviewStatus = T.String(namespace="vnd")

item = ItemDef(OID="IT.DM.RACE", Name="RACE", DataType="text", Length=20,
               SASFieldName="RACE", ReviewStatus="Draft")
item.Description = DEF.Description()
item.Description.TranslatedText.append(DEF.TranslatedText(_content="Race", lang="en"))
item.Origin.append(DEF.Origin(Type="Collected"))

print(item.to_xml_string())

`vnd:ReviewStatus="Draft"` serialized with the namespace declared — no namespace bookkeeping
on our side.

## 2. Use it in a document

Load the minimal DM define from Block 2 (shipped as `data/define_dm_example.xml`), add the
extended RACE variable, and write an extended define.xml:

In [ ]:
loader = LD.ODMLoader(DL.XMLDefineLoader(model_package="define_2_1"))
loader.open_odm_document("../data/define_dm_example.xml")
odm = loader.root()
mdv = odm.Study.MetaDataVersion

mdv.ItemDef.append(item)
mdv.ItemGroupDef[0].ItemRef.append(DEF.ItemRef(ItemOID="IT.DM.RACE", Mandatory="No", OrderNumber=5))

odm.write_xml("output/define_dm_extended.xml")

for line in open("output/define_dm_extended.xml"):
    if "vnd:" in line:
        print(line.strip()[2500:2700])

## 3. Round-trip: the stock model says no

Strict loading with the stock model rejects the unknown attribute — by design; that
strictness is what catches typos in ordinary documents:

In [ ]:
from odmlib.exceptions import OdmlibTypeError

try:
    stock = LD.ODMLoader(DL.XMLDefineLoader(model_package="define_2_1"))
    stock.open_odm_document("output/define_dm_extended.xml")
    stock.root()
except OdmlibTypeError as e:
    print("stock model refuses:", e)

## 4. Round-trip with a local model package

`custom_define/model.py` in this folder (~10 lines) imports the stock model and overrides
`ItemDef` with the extension. `local_model=True` tells the loader to import your package
instead of a bundled model — the extension becomes a first-class, typed attribute:

In [ ]:
import sys
sys.path.insert(0, os.getcwd())    # make ./custom_define importable

ext_loader = LD.ODMLoader(DL.XMLDefineLoader(model_package="custom_define", local_model=True))
ext_loader.open_odm_document("output/define_dm_extended.xml")
ext_odm = ext_loader.root()

race = ext_odm.Study.MetaDataVersion.find("ItemDef", "OID", "IT.DM.RACE")
print("ReviewStatus round-tripped:", race.ReviewStatus)

ext_odm.write_xml("output/define_dm_extended_rt.xml")
print("attribute survives re-serialization:",
      any("vnd:ReviewStatus" in line for line in open("output/define_dm_extended_rt.xml")))

For files you merely *consume* (someone else's extension), permissive mode skips the local
model entirely:

In [ ]:
import odmlib

with odmlib.permissive():
    perm = LD.ODMLoader(DL.XMLDefineLoader(model_package="define_2_1"))
    perm.open_odm_document("output/define_dm_extended.xml")
    perm_odm = perm.root()

race = perm_odm.Study.MetaDataVersion.find("ItemDef", "OID", "IT.DM.RACE")
print("permissive load, stock model:", race.ReviewStatus)

## 5. The honest story on schema validation

The stock Define-XML v2.1 XSD pins each element's attribute set, so a vendor attribute fails
*stock* schema validation — a property of the standard, not of odmlib:

In [ ]:
from odmlib.odm_parser import ODMSchemaValidator

xsd = ODMSchemaValidator(standard="define", version="2.1")
for err in xsd.xsd.iter_errors("output/define_dm_extended.xml"):
    print("stock XSD:", err.reason[:110])

Submission-grade extensions ship an *extension schema* (the standard's `define-extension.xsd`
exists for exactly this), which plugs straight in:

```python
ODMSchemaValidator(xsd_file="your_extended_schema.xsd")
```

The OID and ordering checks work unchanged on extended documents; the conformance checker
validates against the stock model, so expect it to flag extension fields.

For small annotations that must stay valid against the stock schema, use the standard's own
`Alias` element — no extension namespace needed.

**Going further:** `tests/model_extended.py` in the odmlib repo is a complete hand-written
extension model, and odmlib's own `define_2_1` and `arm_1_0` packages are production-scale
examples of exactly this technique.